# Customer Support Ticket Priority Classification

**AI Services MS2 | Advanced Techniques Supplementary B | 60 Marks | Exactly 20 Tests**

**CANDIDATE NOTEBOOK — complete all TODO function bodies.**

Input CSV files are in this folder. Generated model and report files are created only when workflow functions run.

## 1. Imports, parameters and feature groups

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

TARGET = "priority"

CLASS_NAMES = ["HIGH", "LOW", "MEDIUM"]

RAW_NUMERIC = ["response_minutes", "touch_count", "backlog_count", "agent_count", "agent_experience_yrs"]

NUMERIC = RAW_NUMERIC + ["opened_hour", "opened_month", "backlog_per_agent"]

CATEGORICAL = ["channel", "product_area", "customer_tier"]

FEATURES = NUMERIC + CATEGORICAL

## 2. Independently testable functions

Complete `load_data`, `build_pipeline`, `prepare_datasets`, `train_models`, `evaluate_models`, `select_best_model` and the prediction function. Follow exact return types in the instructions.

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Normalize ticket records, calculate time/ratio features, and preserve optional target."""

    df = pd.read_csv(filename,dtype={'ticket_id':'string'})
    df = df.drop_duplicates(subset=['ticket_id'],keep='first')

    df['opened_ts'] = pd.to_datetime(df['opened_ts'],errors='coerce',format='mixed',dayfirst=True)
    df = df.dropna(subset=['opened_ts'])
    df['opened_hour'] = df['opened_ts'].dt.hour
    df['opened_month'] = df['opened_ts'].dt.month

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in CATEGORICAL:
        df[col] = df[col].astype('string').str.strip().str.lower().replace('',pd.NA).replace(r'\.0+$','',regex=True).fillna('unknown')

    df['backlog_per_agent'] = df['backlog_count']/df['agent_count'].where(df['agent_count']>0)

    if TARGET in df:
        df[TARGET] = df[TARGET].astype("string").str.strip().str.upper().replace("", pd.NA)

    return df
    raise NotImplementedError('Complete load_data')

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Return an unfitted mixed-type pipeline with numeric median and categorical mode imputation."""

    numeric_pipeline = Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('numeric',numeric_pipeline,NUMERIC),
        ('category',categorical_pipeline,CATEGORICAL)
    ])

    return Pipeline([('preprocessor',preprocessor),('model',algorithm)])
    raise NotImplementedError('Complete build_pipeline')

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, LabelEncoder]:
    """Return X_train, X_test, integer y_train, integer y_test, fitted label encoder."""

    train_df = load_data(train_file)
    test_df = load_data(test_file)

    encoder = LabelEncoder().fit(CLASS_NAMES)

    X_train = train_df[FEATURES]
    X_test = test_df[FEATURES]

    y_train = pd.Series(encoder.transform(train_df[TARGET]),index=train_df.index,name=TARGET)
    y_test = pd.Series(encoder.transform(test_df[TARGET]),index=test_df.index,name=TARGET)

    return X_train,X_test,y_train,y_test,encoder

    raise NotImplementedError('Complete prepare_datasets')

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Fit two independently preprocessed multiclass classifiers."""

    models = {
        'logistic_regression' : LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        'random_forest' : RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)
    }

    trained_models = {}

    for name,algo in models.items():
        pipeline = build_pipeline(algo)
        pipeline.fit(X_train,y_train)
        trained_models[name] = pipeline

    return trained_models
    raise NotImplementedError('Complete train_models')

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Compare accuracy, macro-F1, weighted-F1 and return confusion matrices for all models."""

    results = []
    class_labels = np.arange(3)
    matrix = {}

    for name,model in trained_models.items():
        predictions = model.predict(X_test)
        results.append({
            'model_name' : name,
            'accuracy' : accuracy_score(y_test,predictions),
            'macro_f1' : f1_score(y_test,predictions,average='macro',zero_division=0,labels=class_labels),
            'weighted_f1' : f1_score(y_test,predictions,average='weighted',zero_division=0,labels=class_labels)
        })
        matrix[name] = confusion_matrix(y_test,predictions,labels=class_labels)

    comparison = pd.DataFrame(results)
    return comparison.sort_values(['macro_f1','weighted_f1','model_name'],ascending=[False,False,True]).reset_index(drop=True),matrix
    raise NotImplementedError('Complete evaluate_models')

In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return first-ranked name and its existing fitted pipeline."""

    best_name = comparison.iloc[0]['model_name']
    best_model = trained_models[best_name]
    return best_name,best_model
    raise NotImplementedError('Complete select_best_model')

In [ ]:
def predict_priority(model: Pipeline, encoder: LabelEncoder, df: pd.DataFrame) -> pd.DataFrame:
    """Predict labels via inverse_transform, confidence and HIGH probability."""
    raise NotImplementedError('Complete predict_priority')

## 3. Independent training and prediction workflows

Training saves the fitted pipeline and comparison. Prediction loads it and never calls `fit`.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train and persist model+encoder, comparison and winning confusion matrix."""
    raise NotImplementedError('Complete train')

In [ ]:
def predict() -> pd.DataFrame:
    """Reload fitted model and label encoder, then write ticket_predictions.csv."""
    raise NotImplementedError('Complete predict')

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
# Uncomment to train after completing the functions:
# best_name, comparison = train()
# display(comparison)

In [ ]:
# Uncomment to predict after training:
# result = predict()
# display(result.head())